In [5]:
# Phase 5 — Local Pragmatic Hybrid (build features locally; no Kaggle dataset needed)

import os, json, gc, math, sys, time, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

# Progress
from tqdm.auto import tqdm

# ML
import lightgbm as lgb
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge

# Try optional deps (RDKit, Mordred, Transformers, PyTorch, PyG)
try:
    from rdkit import RDLogger, Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors
    from rdkit.Chem.Scaffolds import MurckoScaffold
    RDLogger.DisableLog("rdApp.*")
    RDKit_AVAILABLE = True
except Exception:
    RDKit_AVAILABLE = False

try:
    from mordred import Calculator, descriptors
    MORDRED_AVAILABLE = True
except Exception:
    MORDRED_AVAILABLE = False

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
except Exception:
    TORCH_AVAILABLE = False

try:
    from torch_geometric.data import Data, InMemoryDataset
    from torch_geometric.loader import DataLoader
    from torch_geometric.nn import GINEConv, global_mean_pool
    PYG_AVAILABLE = True
except Exception:
    PYG_AVAILABLE = False

try:
    from transformers import AutoTokenizer, AutoModel
    TRANSFORMERS_AVAILABLE = True
except Exception:
    TRANSFORMERS_AVAILABLE = False

# Reproducibility
def seed_everything(seed=42):
    import os, random, numpy as np
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything(42)

# --- Paths ---
DATA_DIR = Path("./")         # train.csv and test.csv live here
ART_DIR  = Path("./artifacts")  # all cached features will be saved here
ART_DIR.mkdir(parents=True, exist_ok=True)

# --- Targets & CV ---
TARGETS = ["Tg", "FFV", "Tc", "Density", "Rg"]
N_SPLITS = 5
USE_GROUPK = True   # GroupKFold by Murcko scaffold if RDKit available

# --- Toggles ---
USE_MORDRED = True      # will auto-disable if Mordred not installed
USE_MORGAN  = True
USE_LM_EMB  = True      # polymer LM embeddings; auto-disables if Transformers not installed
USE_GNN     = True      # GINE MPNN; auto-disables if PyG not installed

# Lightweight timer
class Timer:
    def __init__(self, msg=""): self.msg=msg
    def __enter__(self):
        self.t0=time.time(); 
        if self.msg: print(self.msg)
    def __exit__(self, *_):
        print(f"⏱ {self.msg} done in {time.time()-self.t0:.1f}s")


In [7]:
with Timer("Loading train/test CSVs"):
    train = pd.read_csv(DATA_DIR / "competition/train.csv")
    test  = pd.read_csv(DATA_DIR / "competition/test.csv")

assert "id" in train.columns and "id" in test.columns, "train/test need 'id' column"
assert set(TARGETS).issubset(train.columns), f"Missing targets in train: {set(TARGETS)-set(train.columns)}"
assert "SMILES" in train.columns and "SMILES" in test.columns, "Need SMILES column in both train/test"

print("Rows | train:", len(train), " test:", len(test))
train.head(2)


Loading train/test CSVs
⏱ Loading train/test CSVs done in 0.0s
Rows | train: 7973  test: 3


,id,SMILES,Tg,FFV,Tc,Density,Rg
0,87817,*CC(*)c1ccccc1C(=O)OCCCCCC,NaN,0.374645,0.205667,NaN,NaN
1,106919,*Nc1ccc([C@H](CCC)c2ccc(C3(c4ccc([C@@H](CCC)c5...,NaN,0.370410,NaN,NaN,NaN


In [8]:
def standardize_smiles(s: str):
    if not RDKit_AVAILABLE:
        return s
    if not isinstance(s, str): return None
    s = s.replace("*", "C")
    try:
        mol = Chem.MolFromSmiles(s)
        return Chem.MolToSmiles(mol, canonical=True) if mol else None
    except:
        return None

with Timer("Canonicalizing SMILES"):
    train["SMILES_std"] = train["SMILES"].astype(str).apply(standardize_smiles)
    test["SMILES_std"]  = test["SMILES"].astype(str).apply(standardize_smiles)

# Fallback: if RDKit failed, use original SMILES
train["SMILES_std"] = train["SMILES_std"].fillna(train["SMILES"])
test["SMILES_std"]  = test["SMILES_std"].fillna(test["SMILES"])


Canonicalizing SMILES
⏱ Canonicalizing SMILES done in 1.6s


In [9]:
TAB_TRAIN = ART_DIR / "train_tabular.parquet"
TAB_TEST  = ART_DIR / "test_tabular.parquet"

def rdkit_simple_props(mol):
    # Very cheap scalar props; return dict
    if mol is None: 
        return dict(MolWt=np.nan, TPSA=np.nan, NumHDonors=np.nan, NumHAcceptors=np.nan,
                    NumAromaticRings=np.nan, FractionCSP3=np.nan, HeavyAtomCount=np.nan,
                    NumAliphaticRings=np.nan, BertzCT=np.nan)
    return dict(
        MolWt = Descriptors.MolWt(mol),
        TPSA  = rdMolDescriptors.CalcTPSA(mol),
        NumHDonors = rdMolDescriptors.CalcNumHBD(mol),
        NumHAcceptors = rdMolDescriptors.CalcNumHBA(mol),
        NumAromaticRings = rdMolDescriptors.CalcNumAromaticRings(mol),
        FractionCSP3 = rdMolDescriptors.CalcFractionCSP3(mol),
        HeavyAtomCount = Descriptors.HeavyAtomCount(mol),
        NumAliphaticRings = rdMolDescriptors.CalcNumAliphaticRings(mol),
        BertzCT = Descriptors.BertzCT(mol),
    )

def compute_morgan(mol, radius=2, nBits=2048):
    if mol is None:
        return np.zeros(nBits, dtype=np.uint8)
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
    arr = np.zeros((nBits,), dtype=np.uint8)
    Chem.DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def build_tabular(df, name="train"):
    assert RDKit_AVAILABLE, "RDKit is required to build tabular features"
    smiles = df["SMILES_std"].tolist()
    mols = [Chem.MolFromSmiles(s) if isinstance(s,str) else None for s in smiles]
    rows = []
    morg = []
    use_mordred = USE_MORDRED and MORDRED_AVAILABLE
    calc = Calculator(descriptors, ignore_3D=True) if use_mordred else None

    if use_mordred:
        print(f"Computing Mordred descriptors for {name} (this can take a while)...")
        mordred_df = calc.pandas(mols)  # can be slow; but local
        mordred_df = mordred_df.apply(pd.to_numeric, errors="coerce")
        # Drop constant columns
        mordred_df = mordred_df.loc[:, mordred_df.nunique(dropna=True)>1]
    else:
        mordred_df = None

    print(f"Computing simple RDKit props and Morgan {name}...")
    for mol in tqdm(mols):
        rows.append(rdkit_simple_props(mol))
        if USE_MORGAN:
            morg.append(compute_morgan(mol))
    props_df = pd.DataFrame(rows).reset_index(drop=True)

    if USE_MORGAN:
        morg = np.vstack(morg)
        morg_cols = [f"morgan_{i}" for i in range(morg.shape[1])]
        morg_df = pd.DataFrame(morg, columns=morg_cols)
    else:
        morg_df = pd.DataFrame(index=props_df.index)

    # Combine
    parts = [props_df, morg_df]
    if mordred_df is not None:
        mordred_df = mordred_df.reset_index(drop=True)
        parts.append(mordred_df)

    out = pd.concat(parts, axis=1)
    # Basic cleaning
    out = out.replace([np.inf, -np.inf], np.nan)
    # Drop columns all-NaN or constant
    nunique = out.nunique(dropna=True)
    out = out.loc[:, nunique > 0]

    return out

if TAB_TRAIN.exists() and TAB_TEST.exists():
    print("Loading cached tabular features...")
    train_tab = pd.read_parquet(TAB_TRAIN)
    test_tab  = pd.read_parquet(TAB_TEST)
else:
    assert RDKit_AVAILABLE, "RDKit not found and no cached tabular features present."
    with Timer("Building tabular features (train)"):
        train_tab = build_tabular(train, "train")
        train_tab.to_parquet(TAB_TRAIN)
    with Timer("Building tabular features (test)"):
        test_tab  = build_tabular(test, "test")
        # align test columns to train columns
        test_tab = test_tab.reindex(columns=train_tab.columns, fill_value=np.nan)
        test_tab.to_parquet(TAB_TEST)

print("Tabular shapes:", train_tab.shape, test_tab.shape)


Building tabular features (train)
Computing Mordred descriptors for train (this can take a while)...


100%|██████████| 7973/7973 [06:34<00:00, 20.20it/s]


Computing simple RDKit props and Morgan train...


  0%|          | 0/7973 [00:00<?, ?it/s]

⏱ Building tabular features (train) done in 406.1s


ValueError: Duplicate column names found: ['MolWt', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumAromaticRings', 'FractionCSP3', 'HeavyAtomCount', 'NumAliphaticRings', 'BertzCT', 'morgan_0', 'morgan_1', 'morgan_2', 'morgan_3', 'morgan_4', 'morgan_5', 'morgan_6', 'morgan_7', 'morgan_8', 'morgan_9', 'morgan_10', 'morgan_11', 'morgan_12', 'morgan_13', 'morgan_14', 'morgan_15', 'morgan_16', 'morgan_17', 'morgan_18', 'morgan_19', 'morgan_20', 'morgan_21', 'morgan_22', 'morgan_23', 'morgan_24', 'morgan_25', 'morgan_26', 'morgan_27', 'morgan_28', 'morgan_29', 'morgan_30', 'morgan_31', 'morgan_32', 'morgan_33', 'morgan_34', 'morgan_35', 'morgan_36', 'morgan_37', 'morgan_38', 'morgan_39', 'morgan_40', 'morgan_41', 'morgan_42', 'morgan_43', 'morgan_44', 'morgan_45', 'morgan_46', 'morgan_47', 'morgan_48', 'morgan_49', 'morgan_50', 'morgan_51', 'morgan_52', 'morgan_53', 'morgan_54', 'morgan_55', 'morgan_56', 'morgan_57', 'morgan_58', 'morgan_59', 'morgan_60', 'morgan_61', 'morgan_62', 'morgan_63', '

In [10]:
# Hotfix for already-built DataFrames in memory (train_tab exists; test_tab not built yet)

def uniqueify_columns(df: pd.DataFrame) -> pd.DataFrame:
    cols = pd.Series(df.columns, dtype="object")
    # identify duplicates
    dup_mask = cols.duplicated(keep=False)
    if dup_mask.any():
        # for duplicates, append incremental suffixes based on order
        cols[dup_mask] = (
            cols[dup_mask] + "_" + cols[dup_mask].groupby(cols[dup_mask]).cumcount().astype(str)
        )
    df = df.copy()
    df.columns = cols.values
    return df

# Apply to the in-memory train_tab and save
train_tab = uniqueify_columns(train_tab)
print("Unique train_tab columns:", len(train_tab.columns))
train_tab.to_parquet(TAB_TRAIN)

# Build test_tab now (will still create dups; fix them the same deterministic way)
with Timer("Building tabular features (test)"):
    test_tab  = build_tabular(test, "test")
    test_tab  = uniqueify_columns(test_tab)
    # Align to train columns (missing cols -> NaN)
    test_tab  = test_tab.reindex(columns=train_tab.columns, fill_value=np.nan)
    test_tab.to_parquet(TAB_TEST)

print("Tabular shapes:", train_tab.shape, test_tab.shape)


Unique train_tab columns: 3469
Building tabular features (test)
Computing Mordred descriptors for test (this can take a while)...


100%|██████████| 3/3 [00:00<00:00, 12.74it/s]


Computing simple RDKit props and Morgan test...


  0%|          | 0/3 [00:00<?, ?it/s]

⏱ Building tabular features (test) done in 1.2s
Tabular shapes: (7973, 3469) (3, 3469)


In [11]:
def build_tabular(df, name="train"):
    assert RDKit_AVAILABLE, "RDKit is required to build tabular features"
    smiles = df["SMILES_std"].tolist()
    mols = [Chem.MolFromSmiles(s) if isinstance(s,str) else None for s in smiles]

    # --- Mordred (optional) ---
    use_mordred = USE_MORDRED and MORDRED_AVAILABLE
    if use_mordred:
        print(f"Computing Mordred descriptors for {name} (this can take a while)...")
        calc = Calculator(descriptors, ignore_3D=True)
        mordred_df = calc.pandas(mols).apply(pd.to_numeric, errors="coerce")
        mordred_df = mordred_df.loc[:, mordred_df.nunique(dropna=True) > 1]
        mordred_df = mordred_df.add_prefix("mordred_").reset_index(drop=True)
    else:
        mordred_df = None

    # --- Simple RDKit props (always) ---
    def rdkit_simple_props(mol):
        if mol is None: 
            return dict(MolWt=np.nan, TPSA=np.nan, NumHDonors=np.nan, NumHAcceptors=np.nan,
                        NumAromaticRings=np.nan, FractionCSP3=np.nan, HeavyAtomCount=np.nan,
                        NumAliphaticRings=np.nan, BertzCT=np.nan)
        return dict(
            MolWt = Descriptors.MolWt(mol),
            TPSA  = rdMolDescriptors.CalcTPSA(mol),
            NumHDonors = rdMolDescriptors.CalcNumHBD(mol),
            NumHAcceptors = rdMolDescriptors.CalcNumHBA(mol),
            NumAromaticRings = rdMolDescriptors.CalcNumAromaticRings(mol),
            FractionCSP3 = rdMolDescriptors.CalcFractionCSP3(mol),
            HeavyAtomCount = Descriptors.HeavyAtomCount(mol),
            NumAliphaticRings = rdMolDescriptors.CalcNumAliphaticRings(mol),
            BertzCT = Descriptors.BertzCT(mol),
        )
    props = pd.DataFrame([rdkit_simple_props(m) for m in mols]).add_prefix("rd_")

    # --- Morgan fingerprints (optional) ---
    def compute_morgan(mol, radius=2, nBits=2048):
        if mol is None: return np.zeros(nBits, dtype=np.uint8)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits)
        arr = np.zeros((nBits,), dtype=np.uint8)
        Chem.DataStructs.ConvertToNumpyArray(fp, arr)
        return arr
    if USE_MORGAN:
        morg = np.vstack([compute_morgan(m) for m in mols])
        morg_df = pd.DataFrame(morg, columns=[f"morgan_{i}" for i in range(morg.shape[1])])
    else:
        morg_df = pd.DataFrame(index=props.index)

    # --- Combine & clean ---
    parts = [props, morg_df]
    if mordred_df is not None:
        parts.append(mordred_df)
    out = pd.concat(parts, axis=1).replace([np.inf, -np.inf], np.nan)

    # Remove all-NaN and constant cols
    nunique = out.nunique(dropna=True)
    out = out.loc[:, nunique > 0]

    # Final safety: uniqueify (should be no dups after prefixes, but keep it)
    out = uniqueify_columns(out)
    return out


In [13]:
EMB_TRAIN = ART_DIR / "train_embeddings.npy"
EMB_TEST  = ART_DIR / "test_embeddings.npy"

# Old
# def compute_lm_embeddings(smiles_list, model_name="seyonec/polyBART", batch_size=32, device="cpu"):

# New
def compute_lm_embeddings(smiles_list, model_name="seyonec/ChemBERTa-zinc-base-v1", batch_size=32, device="cpu"):

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    if TORCH_AVAILABLE:
        model.to(device)
        model.eval()
    embs = []
    for i in tqdm(range(0, len(smiles_list), batch_size), desc="LM embeddings"):
        batch = smiles_list[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True)
        if TORCH_AVAILABLE:
            inputs = {k: v.to(device) for k,v in inputs.items()}
            with torch.no_grad():
                out = model(**inputs).last_hidden_state.mean(dim=1).detach().cpu().numpy()
        else:
            # Very slow CPU-only but still works
            out = model(**inputs).last_hidden_state.mean(dim=1).detach().numpy()
        embs.append(out)
    embs = np.vstack(embs).astype(np.float32)
    return embs

if EMB_TRAIN.exists() and EMB_TEST.exists():
    print("Loading cached LM embeddings...")
    train_emb = np.load(EMB_TRAIN)
    test_emb  = np.load(EMB_TEST)
elif USE_LM_EMB and TRANSFORMERS_AVAILABLE:
    device = "cuda" if (TORCH_AVAILABLE and torch.cuda.is_available()) else "cpu"
    with Timer("Computing LM embeddings (train)"):
        train_emb = compute_lm_embeddings(train["SMILES_std"].tolist(), device=device)
        np.save(EMB_TRAIN, train_emb)
    with Timer("Computing LM embeddings (test)"):
        test_emb  = compute_lm_embeddings(test["SMILES_std"].tolist(), device=device)
        np.save(EMB_TEST, test_emb)
else:
    print("Transformers not available or USE_LM_EMB=False — using zeros for LM embeddings.")
    train_emb = np.zeros((len(train), 0), dtype=np.float32)
    test_emb  = np.zeros((len(test),  0), dtype=np.float32)

print("LM embedding shapes:", train_emb.shape, test_emb.shape)


Computing LM embeddings (train)


tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

LM embeddings:   0%|          | 0/250 [00:00<?, ?it/s]

⏱ Computing LM embeddings (train) done in 14.8s
Computing LM embeddings (test)


LM embeddings:   0%|          | 0/1 [00:00<?, ?it/s]

⏱ Computing LM embeddings (test) done in 0.4s
LM embedding shapes: (7973, 768) (3, 768)


In [14]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

with Timer("Assembling feature matrices"):
    imputer_tab = SimpleImputer(strategy="median")
    Xtab_tr = imputer_tab.fit_transform(train_tab.values)
    Xtab_te = imputer_tab.transform(test_tab.values)

    scaler_tab = StandardScaler(with_mean=True, with_std=True)
    scaler_emb = StandardScaler(with_mean=True, with_std=True) if train_emb.shape[1] else None

    Xtab_tr = scaler_tab.fit_transform(Xtab_tr)
    Xtab_te = scaler_tab.transform(Xtab_te)

    if train_emb.shape[1]:
        Xemb_tr = scaler_emb.fit_transform(train_emb)
        Xemb_te = scaler_emb.transform(test_emb)
        X_train = np.concatenate([Xtab_tr, Xemb_tr], axis=1).astype(np.float32)
        X_test  = np.concatenate([Xtab_te, Xemb_te], axis=1).astype(np.float32)
    else:
        X_train = Xtab_tr.astype(np.float32)
        X_test  = Xtab_te.astype(np.float32)

    y_train = train[TARGETS].reset_index(drop=True)

print("Final X shapes:", X_train.shape, X_test.shape)


Assembling feature matrices
⏱ Assembling feature matrices done in 3.5s
Final X shapes: (7973, 4237) (3, 4237)


In [15]:
def murcko_scaffold(smi: str):
    if not RDKit_AVAILABLE: return None
    try:
        mol = Chem.MolFromSmiles(smi)
        core = MurckoScaffold.GetScaffoldForMol(mol)
        return Chem.MolToSmiles(core, isomericSmiles=True)
    except Exception:
        return None

if USE_GROUPK and RDKit_AVAILABLE:
    print("Building Murcko scaffolds for GroupKFold...")
    scaffolds = train["SMILES_std"].astype(str).apply(murcko_scaffold)
    groups = scaffolds.fillna(train["SMILES_std"].astype(str)).astype("category").cat.codes.values
    splitter = GroupKFold(n_splits=N_SPLITS)
    folds = list(splitter.split(X_train, y_train, groups))
else:
    print("Using plain KFold (no RDKit or USE_GROUPK=False).")
    splitter = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
    folds = list(splitter.split(X_train, y_train))

print("Prepared folds:", len(folds))


Building Murcko scaffolds for GroupKFold...
Prepared folds: 5


In [18]:
TARGET_WEIGHTS = {t: 1.0 for t in TARGETS}  # adjust if you have official weights

from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

def weighted_mae(y_true_df: pd.DataFrame, y_pred_df: pd.DataFrame, weights=None):
    w = weights or TARGET_WEIGHTS
    maes = {}
    for t in TARGETS:
        mask = y_true_df[t].notna() & y_pred_df[t].notna()
        if mask.sum() == 0:
            maes[t] = np.nan
        else:
            maes[t] = mean_absolute_error(y_true_df.loc[mask, t].values,
                                          y_pred_df.loc[mask, t].values)
    # normalize by sum of weights for targets that actually had a score
    valid_targets = [t for t in TARGETS if not np.isnan(maes[t])]
    total_w = sum(w.get(t, 1.0) for t in valid_targets) if valid_targets else 1.0
    score = (sum(w.get(t, 1.0) * maes[t] for t in valid_targets) / total_w) if valid_targets else np.nan
    return score, maes

# quick visibility on label completeness
print("Weights:", TARGET_WEIGHTS)
print("Missing label counts per target:", {t: int(train[t].isna().sum()) for t in TARGETS})


Weights: {'Tg': 1.0, 'FFV': 1.0, 'Tc': 1.0, 'Density': 1.0, 'Rg': 1.0}
Missing label counts per target: {'Tg': 7462, 'FFV': 943, 'Tc': 7236, 'Density': 7360, 'Rg': 7359}


In [20]:
# === Cell B — LightGBM training with NaN-safe masking, tqdm + ETA, guardrails, and model saving ===
import os, time, math, json, joblib
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm

# --- params ---
LGB_PARAMS = dict(
    objective="mae",
    learning_rate=0.03,
    num_leaves=127,
    feature_fraction=0.80,
    bagging_fraction=0.80,
    bagging_freq=1,
    min_data_in_leaf=200,
    lambda_l2=1.0,
    max_depth=-1,
    n_estimators=10000,
    random_state=42,
)
MIN_TR = 300      # guardrail: minimum train rows per fold
MIN_VA = 80       # guardrail: minimum valid rows per fold
EARLY_STOP = 300  # early stopping rounds

# --- paths for saving models ---
MODELS_DIR = ART_DIR / "models_lgb"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Initialize OOF with NaN so metrics ignore rows we didn't train/score
oof_lgb  = np.full((len(train), len(TARGETS)), np.nan, dtype=np.float32)
test_lgb = np.zeros((len(test),  len(TARGETS)), dtype=np.float32)

# Label coverage summary (upgrade #1)
coverage = {t: int(train[t].notna().sum()) for t in TARGETS}
print("Label coverage (non-NaN rows):", coverage)

with Timer("Training LightGBM (per target) + ETA + model saving"):
    # tqdm over targets with dynamic total folds for ETA math
    total_effective_folds = 0
    per_target_fold_counts = {}

    # Precompute effective fold counts (after masking+guardrails) to get a meaningful ETA
    for tgt in TARGETS:
        valid_idx = np.where(train[tgt].notna().values)[0]
        if valid_idx.size == 0:
            per_target_fold_counts[tgt] = 0
            continue
        eff = 0
        for (tr_idx, va_idx) in folds:
            tr_mask = np.intersect1d(tr_idx, valid_idx, assume_unique=False)
            va_mask = np.intersect1d(va_idx, valid_idx, assume_unique=False)
            if tr_mask.size >= MIN_TR and va_mask.size >= MIN_VA:
                eff += 1
        per_target_fold_counts[tgt] = eff
        total_effective_folds += eff

    pbar = tqdm(total=total_effective_folds, desc="LGB training (all targets)", leave=True)
    fold_times = []

    for ti, tgt in enumerate(TARGETS):
        print(f"\n▶ Target: {tgt}")
        valid_idx = np.where(train[tgt].notna().values)[0]
        if valid_idx.size == 0:
            print(f"  Skipping {tgt}: no labels present.")
            continue

        # Restrict folds to labeled rows for this target + guardrails (upgrade #2)
        tgt_folds = []
        for (tr_idx, va_idx) in folds:
            tr_mask = np.intersect1d(tr_idx, valid_idx, assume_unique=False)
            va_mask = np.intersect1d(va_idx, valid_idx, assume_unique=False)
            if tr_mask.size >= MIN_TR and va_mask.size >= MIN_VA:
                tgt_folds.append((tr_mask, va_mask))

        if len(tgt_folds) == 0:
            print(f"  Skipping {tgt}: no non-empty folds after masking/guardrails.")
            continue

        test_accum = np.zeros(len(test), dtype=np.float32)
        tgt_fold_models = []  # for saving metadata index

        for f, (tr_idx, va_idx) in enumerate(tgt_folds, start=1):
            t0 = time.time()

            X_tr, X_va = X_train[tr_idx], X_train[va_idx]
            y_tr, y_va = y_train.iloc[tr_idx][tgt].values, y_train.iloc[va_idx][tgt].values

            model = lgb.LGBMRegressor(**LGB_PARAMS)
            model.fit(
                X_tr, y_tr,
                eval_set=[(X_va, y_va)],
                eval_metric="mae",
                callbacks=[lgb.early_stopping(stopping_rounds=EARLY_STOP, verbose=False)]
            )

            # OOF + test preds
            oof_lgb[va_idx, ti] = model.predict(X_va, num_iteration=model.best_iteration_)
            test_accum += model.predict(X_test, num_iteration=model.best_iteration_)

            # Save model (upgrade #3)
            fold_model_path = MODELS_DIR / f"lgb_{tgt}_fold{f}.pkl"
            joblib.dump(model, fold_model_path)
            tgt_fold_models.append(str(fold_model_path))

            # tqdm ETA update
            dt = time.time() - t0
            fold_times.append(dt)
            avg = np.mean(fold_times) if fold_times else dt
            remaining = pbar.total - pbar.n - 1  # after we update below
            eta_sec = max(0.0, remaining * avg)
            pbar.set_postfix({
                "last_fold_s": f"{dt:.1f}",
                "avg_fold_s": f"{avg:.1f}",
                "ETA_min": f"{eta_sec/60:.1f}"
            })
            pbar.update(1)

        # average test preds across effective folds
        test_lgb[:, ti] = test_accum / len(tgt_folds)

        # CV MAE only on valid OOF rows for this target
        mask_valid = ~np.isnan(oof_lgb[:, ti])
        cv_mae = mean_absolute_error(y_train.iloc[mask_valid, :][tgt].values,
                                     oof_lgb[mask_valid, ti]) if mask_valid.any() else np.nan
        print(f"  CV MAE [{tgt}]: {cv_mae:.6f} | folds used: {len(tgt_folds)}")

        # Save a small JSON manifest for this target (models used, cv score, coverage)
        manifest = {
            "target": tgt,
            "folds_used": len(tgt_folds),
            "cv_mae": None if np.isnan(cv_mae) else float(cv_mae),
            "label_count": int(coverage[tgt]),
            "models": tgt_fold_models,
            "params": LGB_PARAMS,
            "guardrails": {"MIN_TR": MIN_TR, "MIN_VA": MIN_VA, "EARLY_STOP": EARLY_STOP},
        }
        with open(MODELS_DIR / f"lgb_{tgt}_manifest.json", "w") as fjson:
            json.dump(manifest, fjson, indent=2)

    pbar.close()

# Wrap into DataFrames for metrics
lgb_cv = pd.DataFrame(oof_lgb, columns=TARGETS)
w, m = weighted_mae(y_train, lgb_cv)
print("\nLightGBM CV wMAE:", round(w, 6), "| per-target:", {k: (None if np.isnan(v) else round(v,6)) for k,v in m.items()})
print("Models saved to:", MODELS_DIR)


Label coverage (non-NaN rows): {'Tg': 511, 'FFV': 7030, 'Tc': 737, 'Density': 613, 'Rg': 614}
Training LightGBM (per target) + ETA + model saving


LGB training (all targets):   0%|          | 0/13 [00:00<?, ?it/s]


▶ Target: Tg
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] lambda_l2 is set=1.0, reg_lambda=0.0 will be ignored. Current value: lambda_l2=1.0
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Warning] min_data_in_leaf is set=200, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=200
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will b

In [24]:
# === Cell: normalized wMAE (IQR-scaled), uses existing OOF ===
import numpy as np, pandas as pd

# lgb_cv is your OOF preds DataFrame; y_train is true labels
def iqr_scaled_wmae(y_true: pd.DataFrame, y_pred: pd.DataFrame):
    maes = {}
    scales = {}
    for t in TARGETS:
        m = y_true[t].notna() & y_pred[t].notna()
        if m.sum() == 0:
            maes[t] = np.nan
            scales[t] = np.nan
            continue
        ae = np.abs(y_true.loc[m, t].values - y_pred.loc[m, t].values)
        mae = ae.mean()
        q1, q3 = np.nanpercentile(y_true.loc[m, t].values, [25, 75])
        iqr = max(1e-8, q3 - q1)
        maes[t] = mae
        scales[t] = iqr
    # normalized by IQR ⇒ lower is better, cross‑target comparable
    norm_vals = [maes[t] / scales[t] for t in TARGETS if not np.isnan(maes[t]) and not np.isnan(scales[t])]
    score = float(np.mean(norm_vals)) if len(norm_vals) else np.nan
    return score, maes, scales

norm_w, raw_mae, scales = iqr_scaled_wmae(y_train, lgb_cv)
print("IQR‑normalized wMAE:", round(norm_w, 6))
print("Per‑target MAE (raw):", {k: (None if np.isnan(v) else round(v,6)) for k,v in raw_mae.items()})
print("Per‑target IQR:", {k: (None if np.isnan(v) else round(v,6)) for k,v in scales.items()})


IQR‑normalized wMAE: 0.481314
Per‑target MAE (raw): {'Tg': 92.107397, 'FFV': 0.008228, 'Tc': 0.077344, 'Density': 0.053992, 'Rg': 2.139518}
Per‑target IQR: {'Tg': 148.462726, 'FFV': 0.031241, 'Tc': 0.144417, 'Density': 0.152667, 'Rg': 3.376874}


In [25]:
# === Cell: post-hoc linear calibration per target, no retraining ===
from sklearn.linear_model import LinearRegression
import numpy as np, pandas as pd

# Load what we need if not in memory
oof = lgb_cv.values.astype(np.float32)  # shape [N, T]
test_preds = np.load(ART_DIR / "test_lgb.npy")     # [Nt, T]

oof_cal  = oof.copy()
test_cal = test_preds.copy()

calib_params = {}

for ti, tgt in enumerate(TARGETS):
    y = y_train[tgt].values
    p = oof[:, ti]
    mask = (~np.isnan(y)) & (~np.isnan(p))
    if mask.sum() < 20:
        calib_params[tgt] = {"strategy": "skip_insufficient_rows"}
        continue

    X = p[mask].reshape(-1, 1)
    lr = LinearRegression().fit(X, y[mask])
    a = float(lr.coef_[0]); b = float(lr.intercept_)
    # Apply to both OOF and test
    oof_cal[mask, ti]  = a * p[mask] + b
    test_cal[:, ti]    = a * test_preds[:, ti] + b
    calib_params[tgt]  = {"a": a, "b": b, "strategy": "linear"}

# Evaluate improvement
oof_cal_df = pd.DataFrame(oof_cal, columns=TARGETS)
w_before, m_before = weighted_mae(y_train, pd.DataFrame(oof, columns=TARGETS))
w_after,  m_after  = weighted_mae(y_train, oof_cal_df)
print("wMAE before:", round(w_before, 6), "after:", round(w_after, 6))
print("by target before:", {k: (None if np.isnan(v) else round(v,6)) for k,v in m_before.items()})
print("by target after :", {k: (None if np.isnan(v) else round(v,6)) for k,v in m_after.items()})
print("calibration params:", calib_params)

# Persist calibrated test predictions (so you can use them for submission/ensembling later)
np.save(ART_DIR / "test_lgb_calibrated.npy", test_cal)
np.save(ART_DIR / "oof_lgb_calibrated.npy", oof_cal)


FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/test_lgb.npy'

In [21]:
G_TRAIN = ART_DIR / "graphs_train.pt"
G_TEST  = ART_DIR / "graphs_test.pt"

def atom_features(atom):
    # basic atom featurization (keep it small)
    Z = atom.GetAtomicNum()
    degree = atom.GetTotalDegree()
    formal = atom.GetFormalCharge()
    hyb = int(atom.GetHybridization())
    aromatic = int(atom.GetIsAromatic())
    in_ring = int(atom.IsInRing())
    return np.array([Z, degree, formal, hyb, aromatic, in_ring], dtype=np.float32)

def bond_features(bond):
    bt = bond.GetBondType()
    onehot = [
        int(bt == Chem.rdchem.BondType.SINGLE),
        int(bt == Chem.rdchem.BondType.DOUBLE),
        int(bt == Chem.rdchem.BondType.TRIPLE),
        int(bt == Chem.rdchem.BondType.AROMATIC),
    ]
    return np.array(onehot + [int(bond.GetIsConjugated()), int(bond.IsInRing())], dtype=np.float32)

def smiles_to_graph(smiles, y=None):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms()==0:
        # minimal 1-node graph
        x = torch.zeros((1,6), dtype=torch.float32)
        edge_index = torch.zeros((2,0), dtype=torch.long)
        edge_attr  = torch.zeros((0,6), dtype=torch.float32)
        data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    else:
        xs = [atom_features(a) for a in mol.GetAtoms()]
        xs = torch.tensor(np.vstack(xs), dtype=torch.float32)
        edges, eattr = [], []
        for b in mol.GetBonds():
            i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
            bf = bond_features(b)
            edges += [[i,j],[j,i]]
            eattr += [bf, bf]
        edge_index = torch.tensor(np.array(edges).T if edges else np.zeros((0,2)).T, dtype=torch.long)
        edge_attr  = torch.tensor(np.vstack(eattr) if eattr else np.zeros((0,6)), dtype=torch.float32)
        data = Data(x=xs, edge_index=edge_index, edge_attr=edge_attr)
    if y is not None:
        data.y = torch.tensor(y, dtype=torch.float32)  # shape [5]
    return data

def build_graph_dataset(df, name="train"):
    data_list = []
    for row in tqdm(df.itertuples(index=False), total=len(df), desc=f"Graphs {name}"):
        smi = getattr(row, df.columns.get_loc("SMILES_std"))
        y = row._asdict() if hasattr(row, "_asdict") else None
        if name=="train":
            labels = [row[df.columns.get_loc(t)] for t in TARGETS]
            g = smiles_to_graph(smi, y=labels)
        else:
            g = smiles_to_graph(smi, y=None)
        data_list.append(g)
    # pack
    data, slices = InMemoryDataset.collate(data_list)
    return {"data": data, "slices": slices}

if USE_GNN and RDKit_AVAILABLE and TORCH_AVAILABLE and PYG_AVAILABLE:
    if G_TRAIN.exists() and G_TEST.exists():
        print("Loading cached graph datasets...")
    else:
        with Timer("Building graph datasets"):
            bundle_tr = build_graph_dataset(train, "train")
            bundle_te = build_graph_dataset(test,  "test")
            torch.save(bundle_tr, G_TRAIN)
            torch.save(bundle_te, G_TEST)

    print("Training a lean GINE MPNN...")
    gs_train = torch.load(G_TRAIN, map_location="cpu")
    gs_test  = torch.load(G_TEST,  map_location="cpu")

    class GraphStore(InMemoryDataset):
        def __init__(self, bundle): 
            self.data, self.slices = bundle["data"], bundle["slices"]
        @property
        def raw_file_names(self): return []
        @property
        def processed_file_names(self): return []

    ds_train = GraphStore(gs_train)
    ds_test  = GraphStore(gs_test)

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    class GINEBlock(nn.Module):
        def __init__(self, in_dim, hidden=128):
            super().__init__()
            self.mlp1 = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.conv1 = GINEConv(self.mlp1)
            self.mlp2 = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
            self.conv2 = GINEConv(self.mlp2)
            self.readout = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU())
        def forward(self, x, edge_index, edge_attr, batch):
            x = F.relu(self.conv1(x, edge_index, edge_attr))
            x = F.relu(self.conv2(x, edge_index, edge_attr))
            x = global_mean_pool(x, batch)
            return self.readout(x)

    class MPNNMulti(nn.Module):
        def __init__(self, in_dim, hidden=128, n_targets=len(TARGETS)):
            super().__init__()
            self.backbone = GINEBlock(in_dim, hidden)
            self.head = nn.Linear(hidden, n_targets)
        def forward(self, data):
            z = self.backbone(data.x, data.edge_index, data.edge_attr, data.batch)
            return self.head(z)

    sample = ds_train[0]
    in_dim = sample.x.size(-1)

    EPOCHS=100; BATCH=64; LR=2e-3; PATIENCE=15
    oof_gnn  = np.zeros((len(train), len(TARGETS)), dtype=np.float32)
    test_gnn = np.zeros((len(test),  len(TARGETS)), dtype=np.float32)

    for f, (tr_idx, va_idx) in enumerate(folds, start=1):
        print(f"\n▶ GNN Fold {f}/{len(folds)}")
        train_list = [ds_train[i] for i in tr_idx]
        valid_list = [ds_train[i] for i in va_idx]
        train_loader = DataLoader(train_list, batch_size=BATCH, shuffle=True)
        valid_loader = DataLoader(valid_list, batch_size=BATCH, shuffle=False)
        test_loader  = DataLoader(ds_test,   batch_size=BATCH, shuffle=False)

        model = MPNNMulti(in_dim=in_dim, hidden=128).to(DEVICE)
        optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
        scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE=="cuda"))

        best = math.inf; bad=0
        for epoch in range(1, EPOCHS+1):
            model.train(); tr_loss=0.0
            for batch in train_loader:
                batch = batch.to(DEVICE)
                yb = batch.y  # [B, 5]
                optim.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(DEVICE=="cuda")):
                    pred = model(batch)
                    loss = F.l1_loss(pred, yb)
                scaler.scale(loss).backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optim); scaler.update()
                tr_loss += loss.item() * yb.size(0)

            # Valid
            model.eval(); va_preds=[]; va_truth=[]
            with torch.no_grad():
                for batch in valid_loader:
                    batch = batch.to(DEVICE)
                    pred = model(batch).detach().cpu().numpy()
                    yb = batch.y.detach().cpu().numpy()
                    va_preds.append(pred); va_truth.append(yb)
            va_preds = np.concatenate(va_preds, axis=0)
            va_truth = np.concatenate(va_truth, axis=0)
            w, _ = weighted_mae(pd.DataFrame(va_truth, columns=TARGETS),
                                pd.DataFrame(va_preds, columns=TARGETS))
            if epoch % 5 == 0 or epoch == 1:
                print(f"  epoch {epoch:03d} | valid wMAE={w:.5f} | best={best:.5f}")
            if w < best - 1e-6:
                best = w; bad=0; best_state = {k:v.cpu() for k,v in model.state_dict().items()}
            else:
                bad += 1
            if bad >= PATIENCE:
                print("  early stopping."); break

        model.load_state_dict(best_state); model.eval()

        # OOF
        va_loader = DataLoader(valid_list, batch_size=BATCH, shuffle=False)
        preds=[]
        with torch.no_grad():
            for batch in va_loader:
                batch = batch.to(DEVICE)
                preds.append(model(batch).detach().cpu().numpy())
        preds = np.concatenate(preds, axis=0)
        oof_gnn[va_idx,:] = preds

        # Test
        tpred=[]; 
        with torch.no_grad():
            for batch in test_loader:
                batch = batch.to(DEVICE)
                tpred.append(model(batch).detach().cpu().numpy())
        tpred = np.concatenate(tpred, axis=0)
        test_gnn += tpred / len(folds)

    gnn_cv = pd.DataFrame(oof_gnn, columns=TARGETS)
    wg, mg = weighted_mae(y_train, gnn_cv)
    print("\nGNN CV wMAE:", round(wg,6), "| per-target:", {k: round(v,6) for k,v in mg.items()})
else:
    print("GNN disabled or deps missing — providing zero predictions as placeholder.")
    oof_gnn  = np.zeros((len(train), len(TARGETS)), dtype=np.float32)
    test_gnn = np.zeros((len(test),  len(TARGETS)), dtype=np.float32)


Building graph datasets


Graphs train:   0%|          | 0/7973 [00:00<?, ?it/s]

⏱ Building graph datasets done in 0.0s


TypeError: getattr(): attribute name must be string

In [22]:
# Stack bases that we actually have
bases = [("lgb", oof_lgb, test_lgb)]
if oof_gnn is not None and oof_gnn.shape[1] == len(TARGETS) and oof_gnn.sum()!=0:
    bases.append(("gnn", oof_gnn, test_gnn))

oof_stack = np.concatenate([b[1] for b in bases], axis=1)
test_stack = np.concatenate([b[2] for b in bases], axis=1)

stack_cols = []
for name, oof, _ in bases:
    stack_cols += [f"{name}_{t}" for t in TARGETS]
oof_stack_df = pd.DataFrame(oof_stack, columns=stack_cols)
test_stack_df = pd.DataFrame(test_stack, columns=stack_cols)

META_PARAMS = dict(
    objective="mae", learning_rate=0.05,
    num_leaves=31, n_estimators=2000,
    feature_fraction=0.9, bagging_fraction=0.9, bagging_freq=1,
    random_state=42,
)

meta_oof  = np.zeros((len(train), len(TARGETS)), dtype=np.float32)
meta_test = np.zeros((len(test),  len(TARGETS)), dtype=np.float32)

with Timer("Training meta-LGBM per target"):
    for ti, tgt in enumerate(TARGETS):
        y = y_train[tgt].values
        for f, (tr_idx, va_idx) in enumerate(folds, start=1):
            Xtr = oof_stack_df.iloc[tr_idx].values
            Xva = oof_stack_df.iloc[va_idx].values
            ytr, yva = y[tr_idx], y[va_idx]

            meta = lgb.LGBMRegressor(**META_PARAMS)
            meta.fit(
                Xtr, ytr,
                eval_set=[(Xva, yva)],
                eval_metric="mae",
                callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
            )
            meta_oof[va_idx, ti] = meta.predict(Xva, num_iteration=meta.best_iteration_)
            meta_test[:, ti]     += meta.predict(test_stack_df.values, num_iteration=meta.best_iteration_) / len(folds)

meta_cv = pd.DataFrame(meta_oof, columns=TARGETS)
wm, mm = weighted_mae(y_train, meta_cv)
print("\nMETA CV wMAE:", round(wm,6), "| per-target:", {k: round(v,6) for k,v in mm.items()})


NameError: name 'oof_gnn' is not defined

In [23]:
def clip_physical(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "FFV" in out:     out["FFV"]     = out["FFV"].clip(lower=0.0, upper=1.0)
    if "Density" in out: out["Density"] = out["Density"].clip(lower=0.2, upper=4.0)
    if "Rg" in out:      out["Rg"]      = out["Rg"].clip(lower=0.0)
    if "Tg" in out:      out["Tg"]      = out["Tg"].clip(lower=0.0)
    if "Tc" in out:      out["Tc"]      = out["Tc"].clip(lower=-300.0, upper=1500.0)
    return out

final_oof  = meta_oof if meta_oof.sum()!=0 else oof_lgb
final_test = meta_test if meta_test.sum()!=0 else test_lgb

final_cv_df = pd.DataFrame(final_oof, columns=TARGETS)
final_score, final_maes = weighted_mae(y_train, final_cv_df)
print("\nFINAL CV wMAE:", round(final_score,6), "| per-target:", {k: round(v,6) for k,v in final_maes.items()})

pred_df = pd.DataFrame(final_test, columns=TARGETS)
pred_df = clip_physical(pred_df)
submission = pd.concat([test[["id"]].reset_index(drop=True), pred_df], axis=1)

# Local save
submission.to_csv("submission.csv", index=False)
pd.DataFrame(final_oof, columns=TARGETS).to_csv("phase5_oof.csv", index=False)
print("Saved:", os.path.abspath("submission.csv"))


NameError: name 'meta_oof' is not defined